# This is our notebook

### Load Data

Here, we load the dataset in using the pandas built in library for reading through csv files. Below is the first 5 rows of the dataset printed to give an idea of the type of data we will be working with.

In [39]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("training_v2.csv")
df.head()


,encounter_id,patient_id,hospital_id,hospital_death,age,bmi,elective_surgery,ethnicity,gender,height,...,aids,cirrhosis,diabetes_mellitus,hepatic_failure,immunosuppression,leukemia,lymphoma,solid_tumor_with_metastasis,apache_3j_bodysystem,apache_2_bodysystem
0,66154,25312,118,0,68.0,22.73,0,Caucasian,M,180.3,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,Sepsis,Cardiovascular
1,114252,59342,81,0,77.0,27.42,0,Caucasian,F,160.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,Respiratory,Respiratory
2,119783,50777,118,0,25.0,31.95,0,Caucasian,F,172.7,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Metabolic,Metabolic
3,79267,46918,118,0,81.0,22.64,1,Caucasian,F,165.1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Cardiovascular,Cardiovascular
4,92056,34377,33,0,19.0,NaN,0,Caucasian,M,188.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Trauma,Trauma


After loading in our data, we handle the missing values and clean it up by encoding categorical variables and define the features and target. Here, we also split the data into 80% training and 20% testing to simulate real-world unseen data.

In [40]:
# Drop ID-like columns (not useful for prediction)
df = df.drop(columns=["encounter_id", "patient_id"], errors="ignore")

# Handle missing values
# numeric fill
num_cols = df.select_dtypes(include=["number"]).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# categorical fill
cat_cols = df.select_dtypes(exclude=["number"]).columns
df[cat_cols] = df[cat_cols].fillna("Unknown")

df = pd.get_dummies(df, drop_first=True)

X = df.drop("hospital_death", axis=1)
y = df["hospital_death"]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_sub = X_train.sample(n = 8000, random_state = 42)
y_train_sub = y_train.loc[X_train_sub.index]

### Baseline Model: Basic SVM

A Support Vector Machine (SVM) with an RBF kernel was used to construct a nonlinear maximum-margin classifier. Given the complexity of the data, we figured this model would be a good starting point for our analysis and prediction of the relationship between these features and the morality rate.

In [41]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

basic_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", probability=True))
])

basic_svm.fit(X_train_sub, y_train_sub)

basic_svm_preds = basic_svm.predict(X_test)
basic_svm_probs = basic_svm.predict_proba(X_test)[:, 1]

### Adding Hyperparameters to Baseline SVM

The effect of hyperparameters ($C$ and $\gamma$) was evaluated to demonstrate changes in model complexity and overfitting behavior.

In [42]:
svm_tuned = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=10, gamma=0.01, probability=True))
])

svm_tuned.fit(X_train_sub, y_train_sub)

svm_tuned_preds = svm_tuned.predict(X_test)
svm_tuned_probs = svm_tuned.predict_proba(X_test)[:, 1]

### Reducing Feature Space

Adding another SVM that has reduced feature space allowed us to analyze the effect of dimensionality on SVM performance. This model uses only 6 selected clinical feaures.

In [43]:
selected_features = [
    "age",
    "bmi",
    "heart_rate_apache",
    "gcs_motor_apache",
    "d1_bun_max",
    "d1_creatinine_max"
]

X_train_reduced = X_train_sub[selected_features]
X_test_reduced = X_test[selected_features]

svm_reduced = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=1, gamma="scale", probability=True))
])

svm_reduced.fit(X_train_reduced, y_train_sub)

svm_reduced_preds = svm_reduced.predict(X_test_reduced)
svm_reduced_probs = svm_reduced.predict_proba(X_test_reduced)[:, 1]

### Accuracies of SVM Models

After creating these models, we decided to test them and compare their accuracies. The performance metric we used for comparison was the Receiver Operating Characteristic Area Under the Curve (ROC-AUC) since it's utilized for binary classification models. This fit perfect with the goal of determining morality (live vs death) through the data. It focuses on separation such that it doesn't have a Pass/Fail cutoff. Instead, it measures how well the model ranks patients. An AUC of 0.88 means that if you randomly pick one patient who died and one who survived, there is an 88% chance the model will give the patient who died a higher risk score.

We chose ROC-AUC as our primary metric because accuracy can be misleading in a hospital setting where deaths are relatively rare. We needed a metric that proved our model could effectively prioritize high-risk patients over low-risk ones, ensuring that the 'signal' of a declining patient wasn't lost in the 'noise' of the surviving majority.

In [44]:
models_1 = {"Basic SVM" : basic_svm_probs, "Tuned SVM": svm_tuned_probs, "Reduced Feature SVM": svm_reduced_probs}

for name, probs in models_1.items():
    auc = roc_auc_score(y_test, probs)
    print(f'{name} - ROC-AUC Score: {auc:.4f}')

Basic SVM - ROC-AUC Score: 0.8525
Tuned SVM - ROC-AUC Score: 0.8416
Reduced Feature SVM - ROC-AUC Score: 0.6200


### Pivot to Logistic Regression

Upon obtaining the ROC-AUC Scores of the SVM models, we were surprised to see the Basic SVM outperformed the Tuned SVM model. We figured that tuning and enforcing more punishment in the Tuned SVM would exhibit better results. When that wasn't the case, we turned to simplifying the model further by changing gears into using a more basic model rather then the complex SVM.

Hence, Logistic Regression was used as a linear probabalistic classifier representing a simple decision boundary. Logistic regression convergence depends on feature scaling and solver choice, which is why Pipeline with StandardScaler and the SAGA solver was used to ensure that convergence on our high-dimensional 91k-row dataset.

In [45]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

baseline_model = Pipeline([("scaler", StandardScaler()), ("lr", LogisticRegression(max_iter = 2000, solver = 'lbfgs'))])

baseline_model.fit(X_train, y_train)

baseline_preds = baseline_model.predict(X_test)
baseline_probs = baseline_model.predict_proba(X_test)[:, 1]

### Model ROC-AUC Comparisons

After implementing the Logistic Regression, we wanted to compute another comparison between the model accuracies with the inclusing of this most recent model to further analyze and understand the relationships of the dataset.

In [46]:

models = {"Logistic Regression": baseline_probs, "Basic SVM" : basic_svm_probs, "Tuned SVM": svm_tuned_probs, "Reduced Reature SVM": svm_reduced_probs}

for name, probs in models.items():
    auc = roc_auc_score(y_test, probs)
    print(f'{name} - ROC-AUC Score: {auc:.4f}')

Logistic Regression - ROC-AUC Score: 0.8832
Basic SVM - ROC-AUC Score: 0.8525
Tuned SVM - ROC-AUC Score: 0.8416
Reduced Reature SVM - ROC-AUC Score: 0.6200


### Analysis of Model Approach:

We initially started with a Support Vector Machine since we figured thatdealing with complex ICU data would require a complex model to approporiately capture the life and death nuances of patient care. We even tried tuning the hyperparameters with the thought that a more 'rigourous' model would perform better.

However, the results surprised us. Not only did our 'Basic' SVM perform better than the 'Tuned' version, but when we tried to reduce the feature space to simplify the model, our accuracy plummeted to a 0.62 AUC. This is barely better than a coin flip. Thus, we concluded two things:

- Complexity isn't always better: Over-tuning led to overfitting.

- Clinical data is rich: You cannot predict mortality by looking at just a few variables; you need the full clinical picture.


Seeing that these more complex models were struggling with the sheer size of our training data of nearly 92,000 patients, we pivoted back to a Logistic Regression. We realized that in the ICU, clinical markers often have a direct, linear relationship with risk. By using a simpler model, we were able to train on the entire dataset instead of a small sample.

After implementing a Logistic Regression model, we were able to achive our highest ROC-AUC of 0.8832, effectively outperforming the complex SVMs while being faster and more interpretable.

Upon further deliberation, we concluded that SVMs can be very sensitive to imbalance, and if it doesn't see enough examples of the minority class (ie patients who passed away), it struggles to find a good boundary. The Basic SVM beat the Tuned SVM because of the overfitting. This caused the tuned model to try too hard to fit the training data perfectly, which made it perform worse on the test data. Especially since it was trained on only about 10% of the patients, it had less 'experience' to effectively learn from.

While SVM is a more complex nonlinear model, the Logistic Regression performed best. This suggests that the predictors for hospital mortality in this dataset have a strong linear relationship with the outcome, and the model benefited significantly from being trained on the full 91k-row dataset rather than a subset.

### Final Model Analysis

With our best-performing model (Logistic Regression, AUC: 0.8832) finalized and validated, we transitioned to the deployment phase. We applied the model to the unlabeled dataset (unseen data) to generate mortality risk probabilities for new patients. To ensure consistency, we performed the exact same preprocessing steps, like handling missing values and aligning features, so the model could interpret the data correctly. The resulting file represents our final clinical predictions, ranking patients by their likelihood of needing urgent intervention.


In [48]:
unlabeled_df = pd.read_csv("unlabeled.csv")

submission_ids = unlabeled_df['encounter_id']

unlabeled_df = unlabeled_df.drop(columns=["encounter_id", "patient_id"], errors="ignore")
train_medians = df[num_cols].median()
unlabeled_df[num_cols] = unlabeled_df[num_cols].fillna(train_medians)

unlabeled_df[cat_cols] = unlabeled_df[cat_cols].fillna("Unknown")
X_unlabeled = pd.get_dummies(unlabeled_df, drop_first = True)

X_unlabeled = X_unlabeled.reindex(columns = X.columns, fill_value = 0)

final_probs = baseline_model.predict_proba(X_unlabeled)[:, 1]

submission = pd.read_csv("solution_template.csv")

submission["hospital_death"] = final_probs
submission.to_csv("my_final_predictions.csv", index = False)

fin = pd.read_csv("my_final_predictions.csv")
fin.head()

,encounter_id,hospital_death
0,2,0.044505
1,5,0.010659
2,7,0.018282
3,8,0.104474
4,10,0.245887
